This notebook performs imputed methylome ~ phenotype associations
- run in batches using SAK
- test CpG ~ delirium / dementia associations
- needs (1) predictions matrix (2) phenotype matrix (3) mwas_03_assoc.sh (4) mwas_03.1_assoc.py
- Date: 06.02.2026


## Setup

In [1]:
%%bash
pip install statsmodels
pip install patsy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 170.7 MB/s eta 0:00:00


In [2]:
import os
import glob
import pandas as pd
from pandas.core.common import flatten
import re
import numpy as np
import seaborn as sns
from tqdm import tqdm
import time
import sys
import statsmodels.api as sm
from scipy.stats import pearsonr

### Methylome wide associations 
- in batches - use SAK 

In [98]:
%%bash
dx upload --brief mwas_03_assoc.sh --dest vasilis/SAK_scripts/
dx upload --brief mwas_03.1_assoc.py --dest vasilis/SAK_scripts/

file-J630zz0JZB70f620Zp6x48xy
file-J630zz8JZB79k3vZ72PxykZB


In [99]:
%%bash

### select phenotype
pheno='dementia_status'
###

phenofile="vasilis/data/ukb_wb_del_cov_e4_dem_alz_phe.txt"
predictdir="vasilis/data/ebb/scores"
dest="vasilis/data/ebb/results/"

for batch in {1..50}; do
    dx run swiss-army-knife \
        -iin="vasilis/SAK_scripts/mwas_03_assoc.sh" \
        -iin="vasilis/SAK_scripts/mwas_03.1_assoc.py" \
        -iin="${predictdir}/batch${batch}_predict.txt" \
        -iin="${phenofile}" \
        -icmd="sh mwas_03_assoc.sh ${batch} ${pheno}" \
        --tag="as_${batch}_${pheno}" \
        --instance-type "mem1_ssd1_v2_x36" \
        --destination="${dest}" \
        --brief --yes --priority high
done


job-J63109jJZB70f620Zp6x490G
job-J6310B0JZB7BKv6BYFxK1PyZ
job-J6310B8JZB7BKv6BYFxK1Pyf
job-J6310BQJZB70f620Zp6x490f
job-J6310BjJZB7BKv6BYFxK1Pyv
job-J6310F0JZB7BKv6BYFxK1Pyz
job-J6310F8JZB79k3vZ72Pxykfj
job-J6310FQJZB79k3vZ72Pxykfp
job-J6310FjJZB79k3vZ72Pxykfv
job-J6310G0JZB7FgFJ53JvPx5yY
job-J6310G8JZB79ggbq7yBgVgPV
job-J6310GQJZB7FgFJ53JvPx5yb
job-J6310GjJZB7BKv6BYFxK1Pz1
job-J6310J0JZB70f620Zp6x490x
job-J6310J8JZB7BKv6BYFxK1PzF
job-J6310JQJZB7BKv6BYFxK1PzJ
job-J6310JjJZB70f620Zp6x4913
job-J6310K8JZB70f620Zp6x4915
job-J6310K8JZB79k3vZ72Pxykj9
job-J6310KjJZB70f620Zp6x4917
job-J6310P0JZB70f620Zp6x4919
job-J6310P8JZB70f620Zp6x491F
job-J6310PQJZB70f620Zp6x491J
job-J6310PjJZB79ggbq7yBgVgPY
job-J6310Q0JZB79k3vZ72PxykjK
job-J6310Q8JZB7FgFJ53JvPx5yp
job-J6310QQJZB7Bx6QJfb4X8v6f
job-J6310QjJZB7BzzxpZg9Qbq5p
job-J6310V0JZB7PZ3Fz9kjzKb2y
job-J6310V8JZB7BzzxpZg9Qbq5y
job-J6310VQJZB7KbYX921GXV11Z
job-J6310VQJZB7PZ3Fz9kjzKb34
job-J6310VjJZB7BzzxpZg9Qbq62
job-J6310X0JZB7KbYX921GXV11x
job-J6310X8JZB

### Manually

In [4]:
!dx download -f vasilis/data/ebb/scores/batch1_predict.txt
!dx download -f vasilis/data/ukb_wb_del_cov_e4_dem_alz_phe.txt

[===========================================================>] Completed 6,125,097,184 of 6,125,097,184 bytes (100%) /opt/notebooks/batch1_predict.txtt


In [5]:
%%bash

batch=1

### setup phenotype
PHENO=delirium

### setup inputs
PHENOFILE_INIT=ukb_wb_del_cov_e4_dem_alz_phe.txt
PHENOFILE=pheno.txt
PREDFILE=batch${batch}_predict.txt

### setup output
OUTFILE=${PHENO}_${batch}_assoc.txt

### match phenotypes and predictions files
# IIDs in prediction matrix
awk '{print $1}' $PREDFILE > temp.IDs.predict.txt

# match phenotypes with prediction IIDs - inner join - order of ids in prediction matrix
awk '
  BEGIN { OFS="\t" }
  FNR==1 && NR==FNR {   
    PCS = ""
    for (i=11; i<=NF; i++)
      PCS = PCS (i==11 ? "" : OFS) $i
    hdr2 = $2 OFS $1 OFS $3 OFS $4 OFS $5 OFS $6 OFS $7 OFS $9 OFS PCS  # pheno file header
    next
  }
  NR==FNR {
    PCS = ""
    for (i=11; i<=NF; i++)
      PCS = PCS (i==11 ? "" : OFS) $i
    b[$1] = $2 OFS $1 OFS $3 OFS $4 OFS $5 OFS $6 OFS $7 OFS $9 OFS PCS # pheno file data
    next
  }
  FNR==1 {  
    print hdr2 # output header
    next
  }
  $1 in b { 
    print b[$1]  # Stream predict file in order
  }
' $PHENOFILE_INIT temp.IDs.predict.txt > $PHENOFILE


In [94]:
#pred = pd.read_csv(predfile, sep="\t", usecols=[0,1,2,3,4,5,6])

In [9]:
## load phenotype file
phenofile = 'pheno.txt'
phe  = pd.read_csv(phenofile, sep="\t")

## load predictions files
batch    = 1 # sys.argv[1]
predfile = 'batch' + str(batch) + '_predict.txt'
pred = pd.read_csv(predfile, sep="\t")

In [10]:
pred.head()

,IID,FID,cg08820315,cg18073198,cg27083176,cg14574044,cg22567250,cg08927008,cg26893147,cg16403185,...,cg23189044,cg18361445,cg04059318,cg08055131,cg07532720,cg18716503,cg07757861,cg17822710,cg09349343,cg00011855
0,1000015,1000015,-3.570,0.00,0.0,-0.019974,0.0,-0.013555,0.000000,0.000000,...,-0.001825,7.4,-0.000236,0.000,0.000000,0.022224,-0.004600,-3.49000,-4.48,0.008761
1,1000027,1000027,0.000,0.00,0.0,-0.018483,0.0,-0.029490,0.000000,0.001953,...,0.047231,0.0,0.001218,-4.670,0.056114,0.002484,-0.002329,0.00000,-2.24,-0.001950
2,1000039,1000039,0.000,3.58,0.0,-0.010973,0.0,0.013080,-0.013073,-0.010368,...,0.000000,3.7,-0.000261,-2.335,0.056114,0.003360,0.002454,-3.65826,-2.24,0.007044
3,1000040,1000040,-3.570,3.58,0.0,-0.023008,0.0,-0.010742,0.000000,0.000000,...,-0.005484,3.7,0.000677,0.000,0.000000,-0.000226,-0.000589,0.00000,-4.48,0.003385
4,1000053,1000053,-1.785,0.00,0.0,-0.019203,0.0,-0.002813,0.000000,0.004072,...,0.000000,3.7,0.001663,0.000,0.139055,0.019666,-0.001729,-3.49000,-2.24,0.005578


In [79]:
print(f"Residualising phenotype {pheno}, batch {batch}:")

covar_names = ['sex', 'age', 'batch'] + [f"PC{i}" for i in range(1, 21)]
pheno = 'delirium'

## Get phenotype residuals
Y_orig  = phe[pheno]
X_covar = phe[covar_names]

# Add intercept
X_covar = sm.add_constant(X_covar)

# Fit model
model0 = sm.Logit(Y_orig, X_covar)
fit0   = model0.fit()

# get residuals
Y_resid = fit0.resid_response

Residualising phenotype delirium, batch 1:
Optimization terminated successfully.
         Current function value: 0.088863
         Iterations 9


In [61]:
## Association analysis
cpgs   = pred.columns[2:].tolist()
n_cpgs = len(cpgs)
print(f"Starting {pheno} ~ CpGs association analysis; batch {batch}; n(CpGs)={n_cpgs}")

res = []

## pheno resid ~ CpG
for i in tqdm(range(n_cpgs)):
    # get CpG i data
    cpg_i   = cpgs[i]
    X_cpg_i = sm.add_constant(pred[cpg_i])
    # Fit model
    model_i = sm.OLS(Y_resid, X_cpg_i).fit()
    # add to results df
    res.append({
        'cpg': cpg_i,
        'beta': model_i.params[cpg_i],
        'se': model_i.bse[cpg_i],
        'lci': model_i.conf_int().loc[cpg_i][0],
        'uci': model_i.conf_int().loc[cpg_i][1],
        'zscore': model_i.params[cpg_i] / model_i.bse[cpg_i],
        'pvalue': model_i.pvalues[cpg_i],
        'n_samples': int(model_i.nobs)
    })

res_df = pd.DataFrame(res)

Starting delirium ~ CpGs association analysis; batch 1; n(CpGs)=1975


100%|██████████| 1975/1975 [01:12<00:00, 27.33it/s]


In [87]:
## Association analysis
cpgs   = pred.columns[2:].tolist()
n_cpgs = len(cpgs)
print(f"Starting {pheno} ~ CpGs association analysis; batch {batch}; n(CpGs)={n_cpgs}")

res = []

y = Y_resid.values

## pheno resid ~ CpG
for i in range(n_cpgs):
    
    # get CpG i data
    cpg_i = cpgs[i]  
    x = pred[cpg_i].values

    # drop missing
    mask = ~np.isnan(x) & ~np.isnan(y)
    x_ = x[mask]
    y_ = y[mask]
    n = len(x_)

    if n < 3:
        continue

    # Fit model
    slope, intercept, r, pval, stderr = stats.linregress(x_, y_)

    # 95% CI using t distribution
    tcrit = stats.t.ppf(0.975, df=n - 2)
    lci = slope - tcrit * stderr
    uci = slope + tcrit * stderr

    res.append({
        "cpg": cpg_i,
        "beta": slope,
        "se": stderr,
        "lci": lci,
        "uci": uci,
        "zscore": slope / stderr,
        "pvalue": pval,
        "r": r,
        "n": n,
    })

res_df = pd.DataFrame(res)


Starting delirium ~ CpGs association analysis; batch 1; n(CpGs)=1975


### Load imputed CpG scores

In [3]:
!dx download -f vasilis/data/ebb/scores/batch1_predict.txt

[===========================================================>] Completed 6,125,097,184 of 6,125,097,184 bytes (100%) /opt/notebooks/batch1_predict.txtt


In [17]:
!wc -l batch1_predict.txt
!head -n1 batch1_predict.txt | tr '\t' '\n' | wc -l
!head -n5 batch1_predict.txt | awk '{print $1,$2,$3,$4,$5}'

407607 batch1_predict.txt
1977
IID FID cg08820315 cg18073198 cg27083176
1000015 1000015 -3.57 0.0 0.0
1000027 1000027 0.0 0.0 0.0
1000039 1000039 0.0 3.58 0.0
1000040 1000040 -3.57 3.58 0.0


### Load phenotypes

In [10]:
!dx download -f vasilis/data/ukb_wb_del_cov_e4_dem_alz_phe.txt

[===========================================================>] Completed 87,320,990 of 87,320,990 bytes (100%) /opt/notebooks/ukb_wb_del_cov_e4_dem_alz_phe.txtxt


In [18]:
!wc -l ukb_wb_del_cov_e4_dem_alz_phe.txt
!head -n5 ukb_wb_del_cov_e4_dem_alz_phe.txt | awk '{print $1,$2,$3,$4,$5,$6,$7,$NF}'

407828 ukb_wb_del_cov_e4_dem_alz_phe.txt
FID IID sex age batch e4_status dementia_status delirium
1000015 1000015 1 82 50 1 0 0
1000053 1000053 0 54.7 19 2 0 0
1000132 1000132 0 71.6 44 0 0 0
1000148 1000148 0 54.3 38 0 0 0


### Methylome wide associations

In [ ]:
%%bash
rm test_association.txt
python3 MetaXcan/software/PrediXcanAssociation.py \
    --expression_file batch1_predict.txt \
    --input_phenos_file pheno.txt \
    --input_phenos_column delirium \
    --output test_association.txt \
    --covariates_file pheno.txt \
    --covariates sex age batch PC1 PC2 PC3 PC4 PC5 PC6 PC7 PC8 PC9 PC10 PC11 PC12 PC13 PC14 PC15 PC16 PC17 PC18 PC19 PC20 \
    --verbosity 9 \
    --mode logistic \
    --throw 2> /dev/null

In [46]:
assoc = pd.read_csv('test_association.txt', sep="\t")
assoc.head()

,gene,effect,se,zscore,pvalue,n_samples,status
0,cg17469265,-0.000666,0.000171,-3.901890,0.000095,407606,NaN
1,cg19557340,0.370904,0.110271,3.363564,0.000769,407606,NaN
2,cg01334972,0.169716,0.051986,3.264620,0.001096,407606,NaN
3,cg21875866,-0.082717,0.025390,-3.257880,0.001123,407606,NaN
4,cg10663215,-0.140305,0.044321,-3.165669,0.001547,407606,NaN


In [51]:
print(assoc.shape)
assoc.sort_values('pvalue')

(1975, 7)


,gene,effect,se,zscore,pvalue,n_samples,status
0,cg17469265,-6.658336e-04,0.000171,-3.901890,0.000095,407606,NaN
1,cg19557340,3.709042e-01,0.110271,3.363564,0.000769,407606,NaN
2,cg01334972,1.697161e-01,0.051986,3.264620,0.001096,407606,NaN
3,cg21875866,-8.271746e-02,0.025390,-3.257880,0.001123,407606,NaN
4,cg10663215,-1.403055e-01,0.044321,-3.165669,0.001547,407606,NaN
...,...,...,...,...,...,...,...
1970,cg11454699,2.378859e-07,0.000102,0.002332,0.998139,407606,NaN
1971,cg26713363,3.480848e-07,0.000188,0.001850,0.998524,407606,NaN
1972,cg24564339,2.017457e-07,0.000119,0.001691,0.998651,407606,NaN
1973,cg19534723,1.262347e-07,0.000128,0.000985,0.999214,407606,NaN


### Merge imputed CpG scores

In [68]:
# list of all CpG names
sspaths = os.listdir('scores/scores/')
cpgs = list({x.replace('.sscore', '').replace('.tmp', '') for x in sspaths})
n_cpgs = len(cpgs)
print(n_cpgs)
cpgs[0:4]

321


['cg06702089', 'cg27168291', 'cg05665882', 'cg09347430']

In [ ]:
# initiate predict.txt matrix: samples x scores
input0 = 'scores/scores/' + cpgs[0] + '.sscore'
pred_mat = pd.read_csv(input0, sep="\t").iloc[:,[0,1]]
pred_mat.columns = ['FID', 'IID']
pred_mat.set_index('IID', drop=True, inplace=True)

for i in range(n_cpgs):
    # get cpg i
    cpg_i = cpgs[i]
    # get score for cpg i
    input_i = 'scores/scores/' + cpg_i + '.sscore'
    try:
        score_i = pd.read_csv(input_i, sep="\t").iloc[:,[0,4]]
    except Exception as e:
        print(f"Skipping {cpg_i} due to error: {e}")
        continue
    score_i.columns = ['IID', cpg_i]
    score_i.set_index('IID', drop=True, inplace=True)
    # add to matrix 
    pred_mat = pred_mat.join(score_i, on='IID', how='inner')


In [116]:
pred_mat.reset_index(inplace=True)
print(pred_mat.shape)
pred_mat.head()

(344304, 322)


,IID,FID,cg06702089,cg27168291,cg05665882,cg09347430,cg21286715,cg04478883,cg10850215,cg26519051,...,cg16579158,cg08523384,cg00017639,cg08710234,cg13077413,cg26864084,cg15821496,cg17043247,cg08951638,cg10058597
0,1000015,1000015,0.008663,-2.275,-0.005068,0.000757,0.0,0.000000,0.0,0.004550,...,0.000964,0.024652,3.605,0.00,0.000574,0.001125,0.012155,0.017150,0.012640,0.00
1,1000027,1000027,0.004331,-2.275,0.001628,0.001541,0.0,-0.002974,0.0,0.014275,...,0.002072,-0.002058,3.605,3.18,0.002064,0.004487,0.007213,0.000000,0.011657,0.00
2,1000039,1000039,0.000000,-4.550,-0.005072,0.000757,0.0,0.007723,0.0,0.000000,...,0.001999,0.002769,3.605,3.18,-0.001651,-0.024878,0.005026,0.018477,0.011657,-1.88
3,1000040,1000040,0.000000,-2.275,0.000000,0.001541,0.0,0.007723,0.0,0.023766,...,0.004168,0.000000,0.000,3.18,-0.002006,-0.010280,0.007213,0.034300,0.000000,0.00
4,1000053,1000053,0.000000,-2.275,-0.005072,0.002872,0.0,0.007723,0.0,0.022850,...,-0.000615,0.011481,3.605,6.36,-0.003293,-0.015357,0.005838,-0.011562,0.008442,0.00


In [118]:
pred_mat.to_csv('test_predict.txt', header=True, index=False, sep='\t')

In [123]:
%%bash
dx download -f vasilis/data/ukb_wb_del_cov_e4_dem_phe.txt
awk '{print $1,$2,$NF}' ukb_wb_del_cov_e4_dem_phe.txt > test.pheno
wc -l test.pheno
head test.pheno

407828 test.pheno
FID IID delirium
1000015 1000015 0
1000053 1000053 0
1000132 1000132 0
1000148 1000148 0
1000163 1000163 0
1000419 1000419 0
1000428 1000428 0
1000435 1000435 0
1000561 1000561 0


In [141]:
pred_mat.set_index('IID', inplace=True)

In [147]:
pheno = pd.read_csv('ukb_wb_del_cov_e4_dem_phe.txt', sep=' ')
pheno.set_index('IID', inplace=True)
pheno2 = pheno[pheno.index.isin(pred_mat.index)]
pheno2.reset_index(inplace=True)

In [156]:
pred_mat.reset_index(inplace=True)

In [163]:
order = pred_mat['IID']

pheno_ordered = (
    pheno2
    .set_index('IID')
    .loc[order]
    .reset_index()
)

In [164]:
pheno_ordered['IID'] == pred_mat['IID']

0         True
1         True
2         True
3         True
4         True
          ... 
344299    True
344300    True
344301    True
344302    True
344303    True
Name: IID, Length: 344304, dtype: bool

In [172]:
pheno_ordered[['IID', 'FID', 'delirium']].to_csv('test.pheno', sep="\t", index=False)